In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType)

spark = SparkSession.builder.appName("bdpa-clickflow-eclothing-recommender").config('spark.sql.shuffle.partitions', "50").getOrCreate()


raw = (
    spark.read
        .option("sep", ";")
        .option("header", "true")
        .csv("../dataset/e_shop_clothing_2008.csv")
        .toDF(
              "year", "month", "day", "order", "country",
        "session_id", "category", "item_id", "colour",
        "location", "photo_type", "price", "price_above_avg", "page"
        )
        .withColumn("year",            F.col("year").cast(IntegerType()))
        .withColumn("month",           F.col("month").cast(IntegerType()))
        .withColumn("day",             F.col("day").cast(IntegerType()))
        .withColumn("order",           F.col("order").cast(IntegerType()))
        .withColumn("country",         F.col("country").cast(IntegerType()))
        .withColumn("session_id",      F.col("session_id").cast(IntegerType()))
        .withColumn("category",        F.col("category").cast(IntegerType()))
        .withColumn("colour",          F.col("colour").cast(IntegerType()))
        .withColumn("location",        F.col("location").cast(IntegerType()))
        .withColumn("photo_type",      F.col("photo_type").cast(IntegerType()))
        .withColumn("price",           F.col("price").cast(IntegerType()))
        .withColumn("price_above_avg", F.col("price_above_avg").cast(IntegerType()))
        .withColumn("page",            F.col("page").cast(IntegerType()))
) 
    

assert raw.count() == 165474
assert raw.filter(F.col("session_id").isNull()).count() == 0
assert raw.filter(F.col("item_id").isNull()).count() == 0
assert raw.select("month").distinct().count() == 5 # april to august
assert raw.select("category").distinct().count() == 4 # trousers, skirts, blouses, sale
assert raw.select("item_id").distinct().count() == 217


print("All validation checks passed.")
raw.printSchema()
raw.show(5)

        


All validation checks passed.
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- category: integer (nullable = true)
 |-- item_id: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- photo_type: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_above_avg: integer (nullable = true)
 |-- page: integer (nullable = true)

+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|year|month|day|order|country|session_id|category|item_id|colour|location|photo_type|price|price_above_avg|page|
+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|2008|    4|  1|    1|     29|         1|       1|    A13|     1|  

In [9]:
session_lengths = (
    raw.groupBy("session_id")
        .agg(F.count("*").alias("n_clicks"))
)

session_lengths.describe("n_clicks").show()

n_single = session_lengths.filter(F.col("n_clicks") == 1).count()

n_total = session_lengths.count() 
print(f"Single click session: {n_single}/{n_total} ({(n_single/n_total * 100)} %)")


# item popularity

item_freq = (
    raw.groupBy('item_id') 
        .agg(F.count("*").alias("views"))
        .orderBy(F.desc("views"))
)

item_freq.show(10)


n_sessions = raw.select("session_id").distinct().count() 
n_items = raw.select("item_id").distinct().count() 
n_pairs = raw.select("session_id", "item_id").distinct().count() 
sparsity = 1 - n_pairs / (n_sessions * n_items)

print(f"Utility matrix: {n_sessions} x {n_items}")
print(f"Observed pairs: {n_pairs}")
print(f"Sparsity: {sparsity:.4%}")


# temporal distribution 
raw.groupBy("month").count().orderBy("month").show() 

raw.groupBy("category").count().orderBy("category").show()

(
    raw.groupBy("session_id", "country")
    .count() 
    .groupBy("country")
    .agg(F.countDistinct("session_id").alias("n_sessions"))
    .orderBy(F.desc("n_sessions"))
    .show(10)
)




+-------+------------------+
|summary|          n_clicks|
+-------+------------------+
|  count|             24026|
|   mean|6.8872887704986265|
| stddev| 8.995160632067192|
|    min|                 1|
|    max|               195|
+-------+------------------+

Single click session: 5042/24026 (20.985598934487637 %)
+-------+-----+
|item_id|views|
+-------+-----+
|     B4| 3579|
|     A2| 3013|
|    A11| 2789|
|     P1| 2681|
|    B10| 2566|
|     A4| 2522|
|    A15| 2489|
|     A5| 2354|
|    A10| 2280|
|     A1| 2265|
+-------+-----+
only showing top 10 rows
Utility matrix: 24026 x 217
Observed pairs: 148345
Sparsity: 97.1547%
+-----+-----+
|month|count|
+-----+-----+
|    4|48199|
|    5|35654|
|    6|32242|
|    7|35231|
|    8|14148|
+-----+-----+

+--------+-----+
|category|count|
+--------+-----+
|       1|49742|
|       2|38408|
|       3|38577|
|       4|38747|
+--------+-----+

+-------+----------+
|country|n_sessions|
+-------+----------+
|     29|     19582|
|      9|      

In [20]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 

w = Window.partitionBy("session_id").orderBy(F.desc("order"))

raw_with_rank = raw.withColumn("click_rank", F.row_number().over(w))

ground_truth = (
    raw_with_rank
    .filter(F.col("click_rank") == 1)
    .select("session_id", "item_id")
)

train_raw = raw_with_rank.filter(F.col("click_rank") > 1)


valid_sessions = (
    train_raw.groupBy("session_id")
        .agg(F.count("*").alias("n_train_clicks"))
             .filter(F.col("n_train_clicks") >= 2)
             .select("session_id")
)

train_raw = train_raw.join(valid_sessions, on="session_id", how="inner")
ground_truth = ground_truth.join(valid_sessions, on="session_id", how="inner")


print(f"Sessions in training: {train_raw.select('session_id').distinct().count()}")
print(f"Ground truth pairs:   {ground_truth.count()}")

Sessions in training: 15664
Ground truth pairs:   15664


In [21]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS 

ALPHA = 40.0

interactions = (
    train_raw
        .groupBy("session_id", "item_id") 
        .agg(F.count("*").alias("click_count"))
        .withColumn("preference", F.lit(1))
        .withColumn("confidence", F.lit(1) + F.lit(ALPHA) * F.col("click_count"))
)


item_indexer = StringIndexer(inputCol='item_id', outputCol="item_idx").fit(interactions)

train_idx = (
    item_indexer.transform(interactions)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

train_idx.cache()


ground_truth_idx = (
    item_indexer.transform(ground_truth)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
    .select("session_idx", "item_idx")
)


als = ALS(
    rank = 50,
    maxIter=15,
    regParam=0.1,
    alpha=ALPHA,
    implicitPrefs=True,
    userCol="session_idx",
    itemCol="item_idx",
    ratingCol="preference",
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(train_idx)

all_sessions = train_idx.select("session_idx").distinct()
raw_recs = als_model.recommendForUserSubset(all_sessions, numItems=10)

recs = (
    raw_recs
    .select(
        F.col("session_idx"),
        F.posexplode("recommendations").alias("rank_0", "rec")
    )
    .select(
        "session_idx",
        (F.col("rank_0") + 1).alias("rank"),
        F.col("rec.item_idx").alias("item_Idx"),
        F.col("rec.rating").alias("score"),
    )
)

print(f"Recommendation rows: {recs.count()}")
recs.show(10)

Recommendation rows: 156640


+-----------+----+--------+----------+
|session_idx|rank|item_Idx|     score|
+-----------+----+--------+----------+
|          3|   1|      22|0.96949005|
|          3|   2|      61| 0.9653262|
|          3|   3|      29|0.96401626|
|          3|   4|     102| 0.9498573|
|          3|   5|      47|0.94657654|
|          3|   6|      66| 0.6985994|
|          3|   7|     100| 0.6512235|
|          3|   8|     105| 0.6280722|
|          3|   9|      58|  0.555074|
|          3|  10|      59|0.48586696|
+-----------+----+--------+----------+
only showing top 10 rows


In [22]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator 

param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [20, 50, 100])
    .addGrid(als.regParam, [0.01, 0.1, 1.0])
    .addGrid(als.alpha, [10.0, 40.0, 80.0])
    .build()
)


evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="preference",
    predictionCol="prediction"
)

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42
)

cv_model = cv.fit(train_idx)
best_model = cv_model.bestModel

print("Best rank:     ", best_model.rank)
print("Best regParam: ", best_model._java_obj.parent().getRegParam())
print("Best alpha:    ", best_model._java_obj.parent().getAlpha())

26/04/25 20:41:23 WARN DAGScheduler: Broadcasting large task binary with size 1038.0 KiB
26/04/25 20:41:23 WARN DAGScheduler: Broadcasting large task binary with size 1080.6 KiB
26/04/25 20:41:24 WARN DAGScheduler: Broadcasting large task binary with size 1039.3 KiB
26/04/25 20:41:24 WARN DAGScheduler: Broadcasting large task binary with size 1123.2 KiB
26/04/25 20:41:24 WARN DAGScheduler: Broadcasting large task binary with size 1081.9 KiB
26/04/25 20:41:24 WARN DAGScheduler: Broadcasting large task binary with size 1165.8 KiB
26/04/25 20:41:25 WARN DAGScheduler: Broadcasting large task binary with size 1124.5 KiB
26/04/25 20:41:25 WARN DAGScheduler: Broadcasting large task binary with size 1208.4 KiB
26/04/25 20:41:25 WARN DAGScheduler: Broadcasting large task binary with size 1167.1 KiB
26/04/25 20:41:25 WARN DAGScheduler: Broadcasting large task binary with size 1038.0 KiB
26/04/25 20:41:26 WARN DAGScheduler: Broadcasting large task binary with size 1251.0 KiB
26/04/25 20:41:26 WAR

Best rank:      20
Best regParam:  1.0
Best alpha:     80.0
